# 🚀 Spaceship Titanic — Predição de Transporte

**Disciplina:** Inteligência Computacional  
**Curso:** Tecnologia em Ciência de Dados — Fatec Jundiaí  
**Professor:** Me. Mateus Guilherme Fuini  

---

## Descrição do Problema

O dataset **Spaceship Titanic** (Kaggle) simula um cenário de ficção científica: uma nave espacial colidiu com uma anomalia espaço-temporal e parte dos passageiros foi transportada para outra dimensão. O objetivo é **prever quais passageiros foram transportados** (`Transported = True/False`) com base em informações como origem, destino, cabine, gastos a bordo e dados demográficos.

- **Tipo de problema:** Classificação binária
- **Variável alvo:** `Transported`
- **Dataset:** [Spaceship Titanic — Kaggle](https://www.kaggle.com/competitions/spaceship-titanic/data)
- **Tamanho:** ~8.700 registros de treino, 14 colunas

---

## Índice

1. [Configuração e Carregamento dos Dados](#1)
2. [Análise Exploratória de Dados (EDA)](#2)
3. [Engenharia de Atributos](#3)
4. [Pré-processamento e Pipeline](#4)
5. [Data Leakage](#5)
6. [Modelagem](#6)
7. [Validação: K-Fold e GridSearchCV](#7)
8. [Avaliação dos Resultados](#8)
9. [Discussão Final](#9)

---
## 1. Configuração e Carregamento dos Dados <a id='1'></a>

In [ ]:
# Instalação (caso necessário no Colab)
# !pip install kaggle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='darkgrid')

print('✅ Bibliotecas carregadas com sucesso!')

In [ ]:
!pip install kaggle -q

from google.colab import files
import os, json

# Faça upload do seu kaggle.json (Settings -> API -> Create New Token)
# Lembre-se de aceitar os termos da competição em kaggle.com/competitions/spaceship-titanic
uploaded = files.upload()
filename = list(uploaded.keys())[0]
creds = json.loads(uploaded[filename].decode('utf-8'))

os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY'] = creds['key']

print(f'✅ Autenticado como: {creds["username"]}')

!kaggle competitions download -c spaceship-titanic
!unzip -o spaceship-titanic.zip

df = pd.read_csv('train.csv')
print(f'\nShape: {df.shape}')
df.head()

In [ ]:
# Visão geral das colunas
df.info()

In [ ]:
# Estatísticas descritivas
df.describe(include='all')

In [ ]:
# Análise estatística detalhada das variáveis numéricas
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
num_cols = spend_cols + ['Age']

stats = df[num_cols].agg(['mean', 'median', 'std', 'min', 'max', 'skew'])
print('=== Estatísticas Descritivas — Variáveis Numéricas ===')
print(stats.round(2).to_string())

print('\n=== Percentual de Zeros nas Colunas de Gastos ===')
for col in spend_cols:
    pct_zero = (df[col] == 0).sum() / df[col].notna().sum() * 100
    print(f'  {col}: {pct_zero:.1f}% zeros')

### Interpretação das Estatísticas Descritivas

**Idade (`Age`):**
- Média próxima de 28 anos, com desvio padrão de ~14 anos — distribuição relativamente jovem.
- Valores entre 0 e 79 anos, sem idades negativas ou implausíveis.

**Gastos a bordo (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`):**
- Todas as colunas apresentam **forte assimetria positiva** (skewness alto): a maioria dos passageiros não gastou nada, mas uma minoria gastou valores muito elevados.
- A **mediana é próxima de zero** em todas, enquanto a **média é bem maior** — confirmando a presença de outliers de alto valor.
- Isso sugere que a mediana é mais adequada do que a média para imputação de valores ausentes nessas colunas.
- O alto percentual de zeros é esperado: passageiros em **CryoSleep** não podem gastar nada a bordo.

**Implicações para o modelo:**
- O escalonamento (`StandardScaler`) é essencial para neutralizar as diferenças de magnitude entre colunas.
- A criação do atributo `TotalSpend` agrega o comportamento de consumo em uma única variável mais informativa.

---
## 2. Análise Exploratória de Dados (EDA) <a id='2'></a>

Nesta seção exploramos a distribuição das variáveis, valores ausentes, outliers e relações entre atributos.

### 2.1 Valores Ausentes

In [ ]:
# Mapa de valores ausentes
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Ausentes': missing, 'Percentual (%)': missing_pct})
missing_df = missing_df[missing_df['Ausentes'] > 0]

print(missing_df)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing_df.index, y=missing_df['Percentual (%)'], palette='rocket')
plt.title('Percentual de Valores Ausentes por Coluna')
plt.xticks(rotation=45)
plt.ylabel('% Ausentes')
plt.tight_layout()
plt.show()

### 2.2 Distribuição da Variável Alvo

In [ ]:
# Distribuição do target
ax = df['Transported'].value_counts().plot(kind='bar', color=['#4e8df5', '#f56b4e'])
plt.title('Distribuição da Variável Alvo: Transported')
plt.xlabel('Transportado')
plt.ylabel('Quantidade')
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + 0.1, p.get_height() + 30))
plt.tight_layout()
plt.show()

print(df['Transported'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

### 2.3 Histogramas — Variáveis Numéricas

In [ ]:
# Colunas de gastos a bordo
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(spend_cols + ['Age']):
    axes[i].hist(df[col].dropna(), bins=40, color='#4e8df5', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribuição: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequência')

plt.suptitle('Histogramas — Variáveis Numéricas', fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Boxplots — Detecção de Outliers

In [ ]:
fig, axes = plt.subplots(1, len(spend_cols), figsize=(16, 5))

for i, col in enumerate(spend_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='#f5c84e')
    axes[i].set_title(col)

plt.suptitle('Boxplots — Gastos a Bordo (outliers visíveis)', fontsize=13)
plt.tight_layout()
plt.show()

> **Observação:** As colunas de gastos possuem forte assimetria positiva e muitos outliers — a maioria dos passageiros não gastou nada, mas alguns gastaram valores muito altos. Isso é esperado no contexto do dataset.

### 2.5 Scatterplot — Gastos vs Transported

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=df.dropna(subset=['Spa', 'RoomService', 'Transported']),
    x='Spa', y='RoomService',
    hue='Transported',
    alpha=0.4,
    palette={True: '#4e8df5', False: '#f56b4e'}
)
plt.title('Spa vs RoomService por Transported')
plt.xlim(0, 5000)
plt.ylim(0, 5000)
plt.tight_layout()
plt.show()

### 2.6 Heatmap de Correlação

In [ ]:
# Convertemos Transported para int temporariamente
df_corr = df[spend_cols + ['Age']].copy()
df_corr['Transported'] = df['Transported'].astype(float)

plt.figure(figsize=(9, 7))
sns.heatmap(
    df_corr.corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm',
    square=True,
    linewidths=0.5
)
plt.title('Heatmap de Correlação')
plt.tight_layout()
plt.show()

### 2.7 Análise de Variáveis Categóricas

In [ ]:
cat_cols = ['HomePlanet', 'Destination', 'CryoSleep', 'VIP']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    data = df.dropna(subset=[col, 'Transported'])
    data[col] = data[col].astype(str)
    transported_rate = data.groupby(col)['Transported'].mean().sort_values()
    transported_rate.plot(kind='barh', ax=axes[i], color='#4e8df5')
    axes[i].set_title(f'Taxa de Transporte por {col}')
    axes[i].set_xlabel('Taxa Média')
    axes[i].axvline(0.5, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Taxa de Transported por Variável Categórica', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Engenharia de Atributos <a id='3'></a>

Criamos novos atributos derivados dos dados originais para enriquecer o modelo.

### Atributos criados:

| Atributo | Origem | Justificativa |
|---|---|---|
| `TotalSpend` | Soma dos 5 gastos a bordo | Passageiros que gastaram mais podem ter padrões diferentes de transporte |
| `Deck` | Extração da coluna `Cabin` (formato `deck/num/side`) | A posição na nave pode influenciar a probabilidade de transporte |
| `Side` | Extração da coluna `Cabin` | Lado da cabine (P/S) pode ter correlação com o transporte |
| `FaixaEtaria` | Agrupamento da coluna `Age` em faixas | Reduz o impacto de outliers e facilita a aprendizagem do modelo |

In [ ]:
def feature_engineering(df):
    df = df.copy()

    # Atributo 1: Gasto total a bordo
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpend'] = df[spend_cols].fillna(0).sum(axis=1)

    # Atributo 2 e 3: Deck e Side extraídos da Cabin
    cabin_split = df['Cabin'].str.split('/', expand=True)
    df['Deck'] = cabin_split[0]   # ex: 'B', 'F', 'G'...
    df['Side'] = cabin_split[2]   # 'P' ou 'S'

    # Atributo 4: Faixa etária
    df['FaixaEtaria'] = pd.cut(
        df['Age'],
        bins=[0, 12, 18, 35, 60, 200],
        labels=['Criança', 'Adolescente', 'Adulto Jovem', 'Adulto', 'Idoso']
    ).astype(str)

    return df

df = feature_engineering(df)

print('Novos atributos criados:')
print(df[['TotalSpend', 'Deck', 'Side', 'FaixaEtaria']].head(10))

In [ ]:
# Verificando distribuição dos novos atributos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['Deck'].value_counts().plot(kind='bar', ax=axes[0], color='#4e8df5')
axes[0].set_title('Distribuição por Deck')

df['Side'].value_counts().plot(kind='bar', ax=axes[1], color='#f56b4e')
axes[1].set_title('Distribuição por Side (P/S)')

df['FaixaEtaria'].value_counts().plot(kind='bar', ax=axes[2], color='#4ef5a8')
axes[2].set_title('Distribuição por Faixa Etária')

plt.tight_layout()
plt.show()

---
## 4. Pré-processamento e Pipeline <a id='4'></a>

Utilizamos `Pipeline` e `ColumnTransformer` do Scikit-learn para organizar todas as transformações de forma segura e reproduzível.

In [ ]:
# Seleção de features e target
FEATURES_NUM = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend']
FEATURES_CAT = ['HomePlanet', 'Destination', 'CryoSleep', 'VIP', 'Deck', 'Side', 'FaixaEtaria']

TARGET = 'Transported'

# Removendo colunas que não usaremos
df_model = df[FEATURES_NUM + FEATURES_CAT + [TARGET]].copy()

# Convertendo booleanos para string (serão tratados como categóricos)
df_model['CryoSleep'] = df_model['CryoSleep'].astype(str)
df_model['VIP'] = df_model['VIP'].astype(str)

# Separando features e target
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET].astype(int)

print(f'X shape: {X.shape}')
print(f'Distribuição do target:\n{y.value_counts()}')

In [ ]:
# Divisão treino/teste ANTES de qualquer transformação (evita data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')

In [ ]:
# Pipeline para variáveis numéricas
# Justificativa: imputação pela mediana é mais robusta a outliers do que a média
# StandardScaler normaliza os dados para algoritmos baseados em distância (KNN) e gradiente
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para variáveis categóricas
# Justificativa: imputação pela moda mantém a distribuição original
# OneHotEncoder converte categorias para representação binária sem hierarquia implícita
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer: aplica cada pipeline às suas respectivas colunas
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, FEATURES_NUM),
    ('cat', categorical_transformer, FEATURES_CAT)
])

print('✅ Preprocessor definido!')

---
## 5. Data Leakage <a id='5'></a>

### O que é Data Leakage?

Data leakage ocorre quando **informações do conjunto de teste "vazam" para o treinamento** do modelo, causando uma estimativa artificialmente otimista de desempenho. O modelo aprende padrões que não estarão disponíveis em produção.

**Exemplos comuns de leakage:**
- Calcular a média para imputação **usando todo o dataset** (treino + teste) antes de dividir
- Fazer normalização **antes** do `train_test_split`
- Usar features derivadas do target

### Como evitamos neste projeto?

1. **Divisão treino/teste PRIMEIRO** — antes de qualquer transformação
2. **Pipeline do Scikit-learn** — o `fit` do pré-processador ocorre apenas nos dados de treino (`fit_transform`), e o teste recebe apenas o `transform`
3. **K-Fold dentro do Pipeline** — o `cross_val_score` e `GridSearchCV` respeitam as dobras e nunca deixam dados de validação "contaminarem" o fit

```
SEM pipeline (leakage):   fit_scaler(X_all) → split → treino/teste
COM pipeline (correto):   split → fit_scaler(X_train) → transform(X_test)
```

---
## 6. Modelagem <a id='6'></a>

In [ ]:
# Pipeline completo: preprocessamento + modelo
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline_knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier())
])

# Treinamento inicial
pipeline_lr.fit(X_train, y_train)
pipeline_knn.fit(X_train, y_train)

print(f'Acurácia Logistic Regression (treino): {pipeline_lr.score(X_train, y_train):.4f}')
print(f'Acurácia KNN             (treino): {pipeline_knn.score(X_train, y_train):.4f}')

---
## 7. Validação: K-Fold e GridSearchCV <a id='7'></a>

### 7.1 K-Fold Cross Validation

O **K-Fold Cross Validation** divide o conjunto de treino em `k` partes (folds). O modelo é treinado em `k-1` folds e avaliado no fold restante, repetindo o processo `k` vezes. A média das acurácias fornece uma estimativa mais confiável do desempenho real do que uma única divisão treino/teste.

Usamos `k=5`, o que significa 5 rodadas com 80%/20% de treino/validação em cada.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores_lr = cross_val_score(pipeline_lr, X_train, y_train, cv=kf, scoring='accuracy')
scores_knn = cross_val_score(pipeline_knn, X_train, y_train, cv=kf, scoring='accuracy')

print('=== K-Fold Cross Validation (k=5) ===')
print(f'Logistic Regression: {scores_lr}')
print(f'  Média: {scores_lr.mean():.4f} | Desvio: {scores_lr.std():.4f}')
print()
print(f'KNN:                 {scores_knn}')
print(f'  Média: {scores_knn.mean():.4f} | Desvio: {scores_knn.std():.4f}')

### 7.2 GridSearchCV — Ajuste de Hiperparâmetros

O **GridSearchCV** testa todas as combinações de parâmetros definidas em um grid, usando Cross Validation para avaliar cada combinação. Escolhemos o modelo com melhor desempenho médio.

#### Parâmetros testados no KNN:
| Parâmetro | Valores | Influência |
|---|---|---|
| `n_neighbors` | 3, 5, 7, 11, 15 | Número de vizinhos considerados — valores baixos = mais sensível ao ruído; valores altos = fronteira mais suave |
| `weights` | uniform, distance | `distance` dá mais peso a vizinhos próximos |
| `metric` | euclidean, manhattan | Forma de calcular a distância entre pontos |

In [ ]:
param_grid_knn = {
    'model__n_neighbors': [3, 5, 7, 11, 15],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    pipeline_knn,
    param_grid_knn,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\n✅ Melhores parâmetros: {grid_search.best_params_}')
print(f'✅ Melhor acurácia (CV): {grid_search.best_score_:.4f}')

In [ ]:
# Visualizando os resultados do GridSearch
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('mean_test_score', ascending=False)
print('Top 10 combinações:')
print(results_df[['param_model__n_neighbors', 'param_model__weights',
                   'param_model__metric', 'mean_test_score', 'std_test_score']].head(10).to_string(index=False))

---
## 8. Avaliação dos Resultados <a id='8'></a>

In [ ]:
# Modelo final: melhor KNN encontrado pelo GridSearch
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'=== Avaliação no Conjunto de Teste ===')
print(f'Acurácia: {acc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Não Transportado', 'Transportado']))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Não Transportado', 'Transportado'])

fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title(f'Matriz de Confusão — KNN Otimizado (Acurácia: {acc:.4f})')
plt.tight_layout()
plt.show()

In [ ]:
# Comparação final entre modelos
resultados = {
    'Logistic Regression (default)': pipeline_lr.score(X_test, y_test),
    'KNN (default)': pipeline_knn.score(X_test, y_test),
    'KNN (GridSearchCV)': acc
}

plt.figure(figsize=(8, 4))
bars = plt.barh(list(resultados.keys()), list(resultados.values()), color=['#4e8df5', '#f5c84e', '#4ef5a8'])
plt.xlabel('Acurácia')
plt.title('Comparação de Modelos — Acurácia no Conjunto de Teste')
plt.xlim(0.5, 1.0)
for bar, val in zip(bars, resultados.values()):
    plt.text(val + 0.002, bar.get_y() + 0.1, f'{val:.4f}', fontsize=11)
plt.tight_layout()
plt.show()

---
## 9. Discussão Final <a id='9'></a>

### Dificuldades encontradas

- Quase todas as colunas do dataset possuem valores ausentes, exigindo uma estratégia de imputação cuidadosa para cada tipo de variável.
- A coluna `Cabin` precisou ser parseada manualmente para extrair informações úteis (`Deck`, `Side`).
- Variáveis com forte assimetria (gastos a bordo) podem prejudicar algoritmos sensíveis à escala.

### Impacto do Pré-processamento

- A imputação pela mediana nas variáveis numéricas foi fundamental para não inflar os valores com outliers.
- O `StandardScaler` foi essencial para o KNN, que é sensível à escala das variáveis.
- A engenharia de atributos (`TotalSpend`, `Deck`, `Side`) agregou informações que o modelo não conseguiria extrair das colunas brutas.

### Limitações do Dataset

- Dataset sintético — pode não refletir complexidades do mundo real.
- Algumas colunas como `Name` e `PassengerId` foram descartadas por não agregarem poder preditivo direto.
- O desequilíbrio de classes é baixo (~50/50), o que favorece a acurácia como métrica principal.

### Possíveis Melhorias Futuras

- Explorar o `PassengerId` para extrair o grupo familiar (os IDs seguem o padrão `XXXX_NN`)
- Testar modelos mais poderosos como `RandomForestClassifier` ou `GradientBoostingClassifier`
- Aplicar `SelectKBest` para seleção de features dentro do pipeline
- Usar `StratifiedKFold` para garantir proporção do target em cada fold